# PicoRV32 CIM SoC — PYNQ 批量验证（20 张图）

**PS 运行时加载 firmware，不需要重新综合 bitstream！**

流程：
1. Overlay 加载 bitstream（一次）
2. 对每张图：hold CPU → 写 firmware hex → release CPU → 读结果
3. 对比 golden model，统计准确率和 bit-exact 率

PS AXI 地址映射：
- 0x4000_0000 (32KB) : FW BRAM — 写入 firmware
- 0x4200_0000 (4KB)  : Result BRAM — 读取推理结果
- 0x4300_0000 (4KB)  : GPIO — bit[0] = cpu_rst_n

上传：`cim_rv32_soc.bit` + `.hwh` + `small_mlp_data/` + `fw_hex_batch/`

In [1]:
from pynq import Overlay, MMIO
import numpy as np
import time, os, glob

## 1. 加载 Bitstream

In [2]:
ol = Overlay('cim_rv32_soc.bit')
print('Overlay loaded!')

Overlay loaded!


## 2. 初始化 MMIO

In [3]:
FW_BASE   = 0x40000000   # FW BRAM (32KB)
RES_BASE  = 0x42000000   # Result BRAM (4KB mapped, 256B used)
GPIO_BASE = 0x43000000   # AXI GPIO (cpu_rst_n)
MAGIC     = 0xC1AA0001

fw_mmio  = MMIO(FW_BASE,  0x8000)   # 32KB
res_mmio = MMIO(RES_BASE, 0x1000)   # 4KB
gpio     = MMIO(GPIO_BASE, 0x1000)  # 4KB (GPIO data at offset 0)

def cpu_hold():
    gpio.write(0x00, 0)   # cpu_rst_n = 0

def cpu_release():
    gpio.write(0x00, 1)   # cpu_rst_n = 1

# Start with CPU held
cpu_hold()
print('CPU held in reset')

CPU held in reset


## 3. Firmware Hex 加载函数

In [4]:
def load_firmware(hex_path):
    """Parse Verilog $readmemh format and write to FW BRAM."""
    with open(hex_path) as f:
        lines = f.readlines()
    
    addr = 0
    for line in lines:
        line = line.strip()
        if not line or line.startswith('//'):
            continue
        if line.startswith('@'):
            addr = int(line[1:], 16)
            continue
        # Data word (hex)
        word = int(line, 16)
        fw_mmio.write(addr * 4, word)  # addr is word-addressed in hex
        addr += 1
    return addr  # number of words written

def run_inference(hex_path, timeout=10):
    """Hold CPU → load firmware → clear result → release → wait magic → read results."""
    cpu_hold()
    time.sleep(0.01)
    
    # Clear result BRAM magic
    res_mmio.write(0x00, 0)
    
    # Load firmware
    n_words = load_firmware(hex_path)
    
    # Release CPU
    cpu_release()
    
    # Wait for magic
    t0 = time.time()
    while True:
        magic = res_mmio.read(0x00)
        if magic == MAGIC:
            break
        if time.time() - t0 > timeout:
            cpu_hold()
            return None  # timeout
        time.sleep(0.05)
    
    elapsed = time.time() - t0
    cpu_hold()
    
    # Read results
    pred = res_mmio.read(0x04)
    expected = res_mmio.read(0x08)
    match = res_mmio.read(0x0C)
    logits = []
    for i in range(10):
        v = res_mmio.read(0x10 + 4*i)
        if v & 0x80000000: v -= 0x100000000
        logits.append(int(v))
    
    return {
        'pred': pred, 'expected': expected, 'match': match,
        'logits': logits, 'elapsed': elapsed, 'n_words': n_words
    }

print('Functions ready.')

Functions ready.


## 4. 读取 Golden 数据

In [5]:
DATA_DIR = 'small_mlp_data'
HEX_DIR  = 'fw_hex_batch'

# Read golden results
golden = {}
gr_path = f'{DATA_DIR}/golden_results.txt'
if os.path.exists(gr_path):
    with open(gr_path) as f:
        for line in f:
            if line.startswith('#'): continue
            parts = line.strip().split()
            if len(parts) >= 4:
                idx = int(parts[0])
                golden[idx] = {'label': int(parts[1]), 'pred': int(parts[2]), 'correct': int(parts[3])}
    print(f'Loaded {len(golden)} golden results')

# Find hex files
hex_files = sorted(glob.glob(f'{HEX_DIR}/firmware_????.hex'))
n_images = len(hex_files)
print(f'Found {n_images} firmware hex files')

# Read golden fc2 outputs for bit-exact check
def read_golden_fc2(idx):
    path = f'{DATA_DIR}/test_images/img_{idx:04d}_fc2.hex'
    if not os.path.exists(path): return None
    with open(path) as f:
        vals = [int(l.strip(), 16) & 0xFF for l in f if l.strip()]
    return np.array([np.uint8(v).view(np.int8) for v in vals], dtype=np.int8)

Loaded 20 golden results
Found 20 firmware hex files


## 5. 逐张推理验证

In [6]:
print(f'Running {n_images} images...\n')

hw_correct = 0
bit_exact = 0
errors = []
total = 0
t_total = time.time()

for hex_path in hex_files:
    idx = int(os.path.basename(hex_path).split('_')[1].split('.')[0])
    g = golden.get(idx, {})
    g_label = g.get('label', '?')
    g_pred = g.get('pred', -1)
    g_fc2 = read_golden_fc2(idx)
    
    result = run_inference(hex_path, timeout=15)
    total += 1
    
    if result is None:
        print(f'  [{idx:04d}] TIMEOUT')
        errors.append(idx)
        continue
    
    # Check
    pred_ok = (result['pred'] == g_pred)
    label_ok = (result['pred'] == g_label)
    logit_ok = True
    if g_fc2 is not None:
        hw_i8 = np.array(result['logits'], dtype=np.int32).astype(np.int8)
        logit_ok = np.array_equal(hw_i8, g_fc2)
    
    if label_ok: hw_correct += 1
    if pred_ok and logit_ok: bit_exact += 1
    else: errors.append(idx)
    
    m = '\u2713' if (pred_ok and logit_ok) else '\u2717'
    c = '\u2713' if label_ok else '\u2717'
    print(f'  [{idx:04d}] label={g_label} hw={result["pred"]} py={g_pred} '
          f'exact={m} correct={c} ({result["elapsed"]:.2f}s)')

t_elapsed = time.time() - t_total
print(f'\nTotal time: {t_elapsed:.1f}s ({t_elapsed/total:.2f}s/image)')

Running 20 images...

  [0000] label=7 hw=7 py=7 exact=✓ correct=✓ (0.05s)
  [0001] label=2 hw=2 py=2 exact=✓ correct=✓ (0.05s)
  [0002] label=1 hw=1 py=1 exact=✓ correct=✓ (0.05s)
  [0003] label=0 hw=0 py=0 exact=✓ correct=✓ (0.05s)
  [0004] label=4 hw=4 py=4 exact=✓ correct=✓ (0.05s)
  [0005] label=1 hw=1 py=1 exact=✓ correct=✓ (0.05s)
  [0006] label=4 hw=4 py=4 exact=✓ correct=✓ (0.05s)
  [0007] label=9 hw=9 py=9 exact=✓ correct=✓ (0.05s)
  [0008] label=5 hw=6 py=6 exact=✓ correct=✗ (0.05s)
  [0009] label=9 hw=9 py=9 exact=✓ correct=✓ (0.05s)
  [0010] label=0 hw=0 py=0 exact=✓ correct=✓ (0.05s)
  [0011] label=6 hw=6 py=6 exact=✓ correct=✓ (0.05s)
  [0012] label=9 hw=9 py=9 exact=✓ correct=✓ (0.05s)
  [0013] label=0 hw=0 py=0 exact=✓ correct=✓ (0.05s)
  [0014] label=1 hw=1 py=1 exact=✓ correct=✓ (0.05s)
  [0015] label=5 hw=5 py=5 exact=✓ correct=✓ (0.05s)
  [0016] label=9 hw=9 py=9 exact=✓ correct=✓ (0.05s)
  [0017] label=7 hw=7 py=7 exact=✓ correct=✓ (0.05s)
  [0018] label=3 hw=8 py

## 6. 结果总结

In [7]:
print('=' * 60)
print('PicoRV32 CIM SoC — Batch Verification')
print('=' * 60)
print(f'  Total images:     {total}')
print(f'  HW accuracy:      {100.*hw_correct/total:.1f}% ({hw_correct}/{total})')
print(f'  Bit-exact match:  {100.*bit_exact/total:.1f}% ({bit_exact}/{total})')
print(f'  Errors/timeouts:  {len(errors)}')
print()
if len(errors) == 0:
    print('>>> ALL IMAGES BIT-EXACT MATCH <<<')
    print('>>> RISC-V controller verified: identical to ARM/Python <<<')
else:
    print(f'  Problem images: {errors}')
print('=' * 60)

PicoRV32 CIM SoC — Batch Verification
  Total images:     20
  HW accuracy:      90.0% (18/20)
  Bit-exact match:  100.0% (20/20)
  Errors/timeouts:  0

>>> ALL IMAGES BIT-EXACT MATCH <<<
>>> RISC-V controller verified: identical to ARM/Python <<<
